In [ ]:
pip install selenium beautifulsoup4


from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from bs4 import BeautifulSoup
import time

def coletar_dados_joli_selenium(url):
    # Configuração do navegador (modo headless opcional)
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(options=options)

    try:
        driver.get(url)
        time.sleep(5)  # aguarda o carregamento da página com JavaScript

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Nome do Produto
        nome_tag = soup.find("h1")
        nome = nome_tag.get_text(strip=True) if nome_tag else "Nome não encontrado"

        # Preço Original
        preco_original_tag = soup.select_one(".joli-joli-theme-1-x-list-price .joli-joli-theme-1-x-price")
        preco_original = preco_original_tag.get_text(strip=True) if preco_original_tag else "Não encontrado"

        # Preço Promocional
        preco_promocional_tag = soup.select_one(".joli-joli-theme-1-x-selling-price .joli-joli-theme-1-x-price")
        preco_promocional = preco_promocional_tag.get_text(strip=True) if preco_promocional_tag else "Não encontrado"

        # Referência
        referencia_tag = soup.find("span", string=lambda text: text and "Referência:" in text)
        referencia = referencia_tag.get_text(strip=True).replace("Referência:", "").strip() if referencia_tag else "Referência não encontrada"

        # Descrição
        descricao_tag = soup.select_one(".joli-joli-theme-1-x-product-description")
        descricao = descricao_tag.get_text("\n", strip=True) if descricao_tag else "Descrição não encontrada"

        # Imagem
        imagem_tag = soup.select_one("img.joli-store-components-0-x-productImageTag--main")
        imagem_url = imagem_tag["src"] if imagem_tag else "Imagem não encontrada"

        # Especificações Técnicas
        especificacoes = {}
        ficha_tecnica_tag = soup.select_one(".joli-joli-theme-1-x-specifications")
        if ficha_tecnica_tag:
            especificacoes_tags = ficha_tecnica_tag.select(".joli-joli-theme-1-x-specification")
            for item in especificacoes_tags:
                chave = item.select_one(".joli-joli-theme-1-x-specificationName")
                valor = item.select_one(".joli-joli-theme-1-x-specificationValue")
                if chave and valor:
                    especificacoes[chave.text.strip()] = valor.text.strip()
        else:
            especificacoes = "Especificações não encontradas"

        # Exibir os dados
        print(f"\n🔹 Nome do Produto: {nome}")
        print(f"💰 Preço Original: {preco_original}")
        print(f"💸 Preço Promocional: {preco_promocional}")
        print(f"🆔 Referência: {referencia}")
        print(f"🖼️ Imagem: {imagem_url}")
        print("\n📄 Descrição:")
        print(descricao)
        print("\n📌 Especificações Técnicas:")
        if isinstance(especificacoes, dict):
            for chave, valor in especificacoes.items():
                print(f"  - {chave}: {valor}")
        else:
            print(especificacoes)

    finally:
        driver.quit()

# Exemplo de uso
url_produto = "https://www.joli.com.br/revestimento-34x60-pei3-lc-avila-bege-hd-cx2-10m2-crie-um-novo-espaco-formigres/p"
coletar_dados_joli_selenium(url_produto)
